### Merge ENA RLPROF-FEX daily files into monthly datasets and reindex them to a continuous 2-min time grid for 2016–2025.
### if you want to get SGP, just replace any 'ENA' to 'SGP'

In [3]:
import xarray as xr
import pandas as pd
import os

# output dir
out_dir = "/data/ggong/ARM_monthly/ENA/rlproffex1thorC1"
os.makedirs(out_dir, exist_ok=True)

# varis we need
vars_to_save = [
    "qc_profile",
    "feature_mask",
    "detection_confidence_score_total",
    "depolarization_ratio",
    "extinction_be",
    "scattering_ratio_e",
    "temperature",
]

# month range
months = pd.date_range("2016-01-01", "2025-12-31", freq="MS")

for t in months:
    year = t.strftime("%Y")
    month = t.strftime("%m")
    
    # input data pattern
    pattern = f"/data/shared_data/ARM_data/ENA/enarlproffex1thorC1/enarlproffex1thorC1.c0.{year}{month}*.nc"
    
    try:
        # read monthly data
        ds = xr.open_mfdataset(pattern, combine="by_coords")
        
        # height limit
        ds_sel = ds.sel(height_high=slice(0.03, 8.0))
        
        # pick varis
        ds_out = ds_sel[vars_to_save]
        
        # output
        out_file = os.path.join(out_dir, f"enarlproffex1thorC1.c0.{year}{month}.nc")
        
        # save
        ds_out.to_netcdf(out_file)
        print(f"✔ Saved: {out_file}")
    
    except Exception as e:
        print(f"✘ Failed {year}-{month}: {e}")


✔ Saved: /data/ggong/ARM_monthly/ENA/rlproffex1thorC1/enarlproffex1thorC1.c0.201601.nc
✔ Saved: /data/ggong/ARM_monthly/ENA/rlproffex1thorC1/enarlproffex1thorC1.c0.201602.nc
✔ Saved: /data/ggong/ARM_monthly/ENA/rlproffex1thorC1/enarlproffex1thorC1.c0.201603.nc
✔ Saved: /data/ggong/ARM_monthly/ENA/rlproffex1thorC1/enarlproffex1thorC1.c0.201604.nc
✘ Failed 2016-05: no files to open
✘ Failed 2016-06: no files to open
✘ Failed 2016-07: no files to open
✘ Failed 2016-08: no files to open
✘ Failed 2016-09: no files to open
✘ Failed 2016-10: no files to open
✘ Failed 2016-11: no files to open
✘ Failed 2016-12: no files to open
✔ Saved: /data/ggong/ARM_monthly/ENA/rlproffex1thorC1/enarlproffex1thorC1.c0.201701.nc
✔ Saved: /data/ggong/ARM_monthly/ENA/rlproffex1thorC1/enarlproffex1thorC1.c0.201702.nc
✔ Saved: /data/ggong/ARM_monthly/ENA/rlproffex1thorC1/enarlproffex1thorC1.c0.201703.nc
✔ Saved: /data/ggong/ARM_monthly/ENA/rlproffex1thorC1/enarlproffex1thorC1.c0.201704.nc
✔ Saved: /data/ggong/ARM

In [14]:
ds_a = xr.open_mfdataset('/data/ggong/ARM_monthly/ENA/rlproffex1thorC1/*.nc')

In [ ]:
full_time = pd.date_range("2016-01-01", "2025-12-31 23:59:00", freq="2min")
ds_a = ds_a.reindex(time=full_time)

In [ ]:
ds_a = ds_a.where(ds_a != -9999.0)

# 2. Remove conflicting attributes (to avoid errors when writing files)
for v in ds_a.data_vars:
    ds_a[v].attrs.pop("missing_value", None)
    ds_a[v].attrs.pop("_FillValue", None)

# 3. save
ds_a.to_netcdf("/data/ggong/ARM_monthly/ENA/rlprof_resample_2min.nc")